# RQ3b — Self-Reflection Intervention
Uses Round 3 (`judgment_after_reflection`) as the final answer. Compares R1→R3 against RQ1 base (R1→R2).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import json
from pathlib import Path
from collections import defaultdict

RQ3B_DIR = Path("../results_todos/rq3b")
RQ1_DIR  = Path("../results_todos/rq1")

RQ1_CONDITIONS = [
    "all_correct_absent", "all_correct_present",
    "mixed_absent",       "mixed_present",
    "all_wrong_absent",   "all_wrong_present",
]
CONDITION_LABELS = {
    "all_correct_absent":  "All-correct\n(no auth)",
    "all_correct_present": "All-correct\n(auth)",
    "mixed_absent":        "Mixed\n(no auth)",
    "mixed_present":       "Mixed\n(auth)",
    "all_wrong_absent":    "All-wrong\n(no auth)",
    "all_wrong_present":   "All-wrong\n(auth)",
}

def compute_rq3b_metrics(jsonl_path):
    """
    Computes metrics using judgment_after_reflection (R3) as final answer.
    - revised_r2: R1 -> R2 change (did peers cause revision in round 2?)
    - revised_r3: R1 -> R3 net change (did the model end up different from start?)
    - harmful/beneficial: based on R1 -> R3 net outcome.
    - FIX: confidence_after_reflection is NOT fallen back to confidence_after.
      Rows missing R3 confidence are skipped entirely to avoid mixing metrics.
    """
    buckets = defaultdict(lambda: {
        "n": 0, "r1": 0, "r3": 0,
        "revised_r2": 0, "revised_r3": 0,
        "cb_n": 0, "harmful": 0, "wb_n": 0, "beneficial": 0, "dc": []
    })
    skipped = 0
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            row = json.loads(line)
            gold = row["answer"]["label"]
            a0 = row.get("judgment_before")
            a2 = row.get("judgment_after")
            a3 = row.get("judgment_after_reflection")
            c0 = row.get("confidence_before")
            # FIX: do NOT fall back to confidence_after — skip if R3 confidence missing
            c3 = row.get("confidence_after_reflection")
            cond = row.get("condition_id", "unknown")
            if any(v is None for v in [a0, a3, c0, c3]):
                skipped += 1
                continue
            b = buckets[cond]
            b["n"] += 1
            r1_ok = (a0 == gold)
            r3_ok = (a3 == gold)
            if r1_ok: b["r1"] += 1
            if r3_ok: b["r3"] += 1
            if a2 is not None and a2 != a0: b["revised_r2"] += 1
            if a3 != a0: b["revised_r3"] += 1
            if r1_ok:
                b["cb_n"] += 1
                if not r3_ok: b["harmful"] += 1
            else:
                b["wb_n"] += 1
                if r3_ok: b["beneficial"] += 1
            b["dc"].append(c3 - c0)
    return buckets, skipped

def pct(n, d): return round(100*n/d, 2) if d > 0 else None
def mn(v): return round(sum(v)/len(v), 4) if v else None

rows_all = []
for jsonl_path in sorted(RQ3B_DIR.rglob("*.jsonl")):
    with open(jsonl_path, encoding="utf-8") as f:
        first = f.readline()
    if "judgment_after_reflection" not in first: continue
    parts = jsonl_path.parts
    try:
        seed_idx = next(i for i, p in enumerate(parts) if p.startswith("seed_"))
        seed = parts[seed_idx].replace("seed_", "")
        model = parts[seed_idx - 1]
    except: continue
    buckets, skipped = compute_rq3b_metrics(jsonl_path)
    print(f"{model} seed_{seed}: {skipped} rows skipped (missing R3 confidence)")
    for cond, b in buckets.items():
        if cond not in RQ1_CONDITIONS: continue
        rows_all.append({
            "model": model, "seed": seed, "condition": cond,
            "n": b["n"],
            "acc_r1": pct(b["r1"], b["n"]),
            "acc_r3": pct(b["r3"], b["n"]),
            "delta_acc": round((b["r3"]-b["r1"])/b["n"]*100, 2) if b["n"] else None,
            "rev_pct_r2": pct(b["revised_r2"], b["n"]),
            "rev_pct_r3": pct(b["revised_r3"], b["n"]),
            "harm_pct": pct(b["harmful"], b["cb_n"]),
            "ben_pct": pct(b["beneficial"], b["wb_n"]),
            "mean_dC": mn(b["dc"]),
        })

combined = pd.DataFrame(rows_all)
combined = combined.groupby(["model", "condition"]).mean(numeric_only=True).reset_index()

def load_rq1(rq1_dir):
    dfs = []
    for csv_path in sorted(rq1_dir.rglob("*.metrics.csv")):
        if "primevul_dataset" in csv_path.name or "_all_seeds" in csv_path.name:
            continue
        parts = csv_path.parts
        if not any(p.startswith("seed_") for p in parts): continue
        df = pd.read_csv(csv_path)
        df = df[df["condition"].isin(RQ1_CONDITIONS)]
        if not df.empty: dfs.append(df)
    all_df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    return all_df.groupby(["model", "condition"]).mean(numeric_only=True).reset_index() if not all_df.empty else pd.DataFrame()

rq1 = load_rq1(RQ1_DIR)
MODELS = sorted(combined["model"].unique())
PALETTE = sns.color_palette("tab10", len(MODELS))
MODEL_COLORS = dict(zip(MODELS, PALETTE))

print(f"\nModels: {MODELS}")
print(f"Conditions: {sorted(combined['condition'].unique())}")


## Per-model tables (Self-reflect, Round 1 vs Round 3)

> **rev_pct_r2** = % of instances where the model revised in Round 2 (after seeing peers). **rev_pct_r3** = % where the net R1→R3 answer differs. A model that revised in R2 but then reverted in R3 contributes to rev_pct_r2 but NOT rev_pct_r3.

In [ ]:
display_cols = ["condition", "n", "acc_r1", "acc_r3", "delta_acc",
                "rev_pct_r2", "rev_pct_r3", "harm_pct", "ben_pct", "mean_dC"]

for model in MODELS:
    df_m = combined[(combined["model"] == model) & (combined["condition"].isin(RQ1_CONDITIONS))].copy()
    present = [c for c in RQ1_CONDITIONS if c in df_m["condition"].values]
    df_m["condition"] = pd.Categorical(df_m["condition"], categories=present, ordered=True)
    df_m = df_m.sort_values("condition")
    avail = [c for c in display_cols if c in df_m.columns]
    df_m = df_m[avail].reset_index(drop=True)
    fmt_cols = [c for c in ["acc_r1","acc_r3","delta_acc","rev_pct_r2","rev_pct_r3","harm_pct","ben_pct","mean_dC"] if c in df_m.columns]
    print(f"\n{'─'*60}\n  {model.upper()} — Self-Reflect\n{'─'*60}")
    display(df_m.style
        .format({"n": lambda x: f"{int(x)}" if pd.notna(x) and x == int(x) else (f"{x:.2f}" if pd.notna(x) else "N/A"),
                 **{c: "{:.1f}" for c in fmt_cols}}, na_rep="N/A")
        .background_gradient(subset=["delta_acc"], cmap="RdYlGn", vmin=-30, vmax=30)
        .background_gradient(subset=["harm_pct"],  cmap="Reds",   vmin=0,   vmax=100)
        .background_gradient(subset=["ben_pct"],   cmap="Greens", vmin=0,   vmax=100)
        .set_caption(f"{model} Self-Reflect"))


## Self-Reflect vs Base: Δ Accuracy comparison

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(5*len(MODELS), 5), sharey=True)
if len(MODELS) == 1:
    axes = [axes]

for ax, model in zip(axes, MODELS):
    for df, label, color in [(rq1, "Base", "#4878d0"), (combined, "Self-Reflect", "#e377c2")]:
        df_m = df[(df["model"] == model) & (df["condition"].isin(RQ1_CONDITIONS))].copy()
        p = [c for c in RQ1_CONDITIONS if c in df_m["condition"].values]
        df_m["condition"] = pd.Categorical(df_m["condition"], categories=p, ordered=True)
        df_m = df_m.sort_values("condition")
        x = range(len(df_m))
        offset = -0.175 if label == "Base" else 0.175
        ax.bar([i + offset for i in x], df_m["delta_acc"], 0.35, label=label, color=color, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(model, fontsize=12, fontweight="bold")
    ax.set_xticks(list(range(len(p))))
    ax.set_xticklabels([CONDITION_LABELS.get(c, c) for c in p], fontsize=8)
    ax.set_ylabel("\u0394 Accuracy (pp)" if ax == axes[0] else "")
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v:+.0f}pp"))

plt.suptitle("Base vs Self-Reflect: \u0394 Accuracy by Condition", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RQ3B_DIR / "rq3b_selfreflect_vs_base.png", dpi=150, bbox_inches="tight")
plt.show()


## Δ Accuracy heatmap — Self-Reflect across models × conditions

In [ ]:
pivot = combined[combined["condition"].isin(RQ1_CONDITIONS)].pivot_table(
    index="condition", columns="model", values="delta_acc")
pivot = pivot.reindex([c for c in RQ1_CONDITIONS if c in pivot.index])

fig, ax = plt.subplots(figsize=(max(6, 2.5*len(MODELS)), 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn", center=0, vmin=-55, vmax=55,
            linewidths=0.5, ax=ax, annot_kws={"size": 10})
ax.set_title("Self-Reflect \u0394 Accuracy (R1\u2192R3) by Model and Condition", fontsize=13, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_yticklabels([CONDITION_LABELS.get(c, c).replace("\n", " ") for c in pivot.index], rotation=0)
plt.tight_layout()
plt.savefig(RQ3B_DIR / "rq3b_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
